# Preprocessing Pipeline

**Complete preprocessing for three DICOM datasets**

## Objectives:
1. Load DICOM series from all three datasets
2. Convert to Hounsfield Units (HU)
3. Resample to uniform spacing
4. Normalize intensity ranges
5. Create initial stent masks
6. Save preprocessed data

In [ ]:
import os
import numpy as np
import pydicom
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from scipy import ndimage
from skimage import filters, morphology
import warnings
warnings.filterwarnings('ignore')

# Load configuration from setup
with open('/content/pipeline_config.json', 'r') as f:
    PIPELINE_CONFIG = json.load(f)

print("🔧 Preprocessing pipeline initialized!")

In [ ]:
class DICOMPreprocessor:
    """Advanced DICOM preprocessing for medical images"""
    
    def __init__(self):
        self.spacing = None
        self.origin = None
        self.direction = None
        print("🔧 DICOM Preprocessor initialized!")
    
    def load_dicom_series(self, folder_path):
        """Load DICOM series from folder"""
        print(f"📂 Loading DICOM series from {folder_path}")
        
        # Get all DICOM files
        dicom_files = [f for f in os.listdir(folder_path) if f.endswith('.dcm')]
        dicom_files.sort()
        
        if not dicom_files:
            raise ValueError(f"No DICOM files found in {folder_path}")
        
        # Read first file to get metadata
        first_slice = pydicom.dcmread(os.path.join(folder_path, dicom_files[0]))
        
        # Get image dimensions
        rows, cols = first_slice.Rows, first_slice.Columns
        num_slices = len(dicom_files)
        
        # Initialize volume
        volume = np.zeros((rows, cols, num_slices), dtype=np.float32)
        
        # Load all slices
        slice_positions = []
        
        for i, filename in enumerate(tqdm(dicom_files, desc="Loading DICOMs")):
            filepath = os.path.join(folder_path, filename)
            ds = pydicom.dcmread(filepath)
            
            # Get slice position
            if hasattr(ds, 'ImagePositionPatient'):
                slice_position = float(ds.ImagePositionPatient[2])
            else:
                slice_position = i
            
            slice_positions.append(slice_position)
            
            # Store pixel data
            volume[:, :, i] = ds.pixel_array.astype(np.float32)
        
        # Sort slices by position
        sort_indices = np.argsort(slice_positions)
        volume = volume[:, :, sort_indices]
        slice_positions = [slice_positions[i] for i in sort_indices]
        
        # Calculate spacing
        if hasattr(first_slice, 'PixelSpacing'):
            pixel_spacing = [float(first_slice.PixelSpacing[0]), float(first_slice.PixelSpacing[1])]
        else:
            pixel_spacing = [1.0, 1.0]
        
        if len(slice_positions) > 1:
            slice_thickness = abs(slice_positions[1] - slice_positions[0])
        else:
            slice_thickness = 1.0
        
        self.spacing = (pixel_spacing[0], pixel_spacing[1], slice_thickness)
        
        print(f"✅ DICOM series loaded: {volume.shape}")
        print(f"📊 Spacing: {self.spacing}")
        
        return volume, slice_positions
    
    def convert_to_hu(self, volume, ds=None):
        """Convert pixel values to Hounsfield Units"""
        print("🔄 Converting to Hounsfield Units...")
        
        # Default values for CT if not available
        intercept = -1024 if ds is None else float(ds.RescaleIntercept) if hasattr(ds, 'RescaleIntercept') else -1024
        slope = 1 if ds is None else float(ds.RescaleSlope) if hasattr(ds, 'RescaleSlope') else 1
        
        # Apply linear transformation
        hu_volume = volume * slope + intercept
        
        print(f"✅ HU conversion completed")
        print(f"📊 HU range: [{hu_volume.min():.1f}, {hu_volume.max():.1f}]")
        
        return hu_volume
    
    def normalize_volume(self, volume, window_center=None, window_width=None):
        """Normalize volume using windowing"""
        print("🔄 Normalizing volume...")
        
        # Default window for soft tissue
        if window_center is None:
            window_center = 40
        if window_width is None:
            window_width = 400
        
        # Calculate window bounds
        window_min = window_center - window_width // 2
        window_max = window_center + window_width // 2
        
        # Apply windowing
        windowed = np.clip(volume, window_min, window_max)
        
        # Normalize to [0, 1]
        normalized = (windowed - window_min) / (window_max - window_min)
        
        print(f"✅ Volume normalized")
        print(f"📊 Normalized range: [{normalized.min():.3f}, {normalized.max():.3f}]")
        
        return normalized
    
    def create_stent_mask(self, hu_volume, hu_threshold=2000):
        """Create initial stent mask using thresholding"""
        print(f"🎯 Creating stent mask (threshold: {hu_threshold} HU)...")
        
        # Threshold for high-density materials (stent)
        binary_mask = hu_volume > hu_threshold
        
        # Morphological operations to clean up
        kernel_size = PIPELINE_CONFIG['preprocessing']['morphology_kernel_size']
        kernel = morphology.disk(kernel_size)
        
        # Remove small objects
        binary_mask = morphology.remove_small_objects(binary_mask, min_size=50)
        
        # Fill holes
        binary_mask = ndimage.binary_fill_holes(binary_mask)
        
        # Morphological opening and closing
        binary_mask = morphology.binary_opening(binary_mask, kernel)
        binary_mask = morphology.binary_closing(binary_mask, kernel)
        
        stent_voxels = np.sum(binary_mask)
        print(f"✅ Stent mask created: {stent_voxels} voxels")
        
        return binary_mask.astype(np.uint8)
    
    def resample_volume(self, volume, target_spacing=(1.0, 1.0, 1.0)):
        """Resample volume to target spacing"""
        print(f"🔄 Resampling to {target_spacing}...")
        
        # Calculate scaling factors
        scale_factors = [
            self.spacing[0] / target_spacing[0],
            self.spacing[1] / target_spacing[1],
            self.spacing[2] / target_spacing[2]
        ]
        
        # Calculate new shape
        new_shape = [
            int(volume.shape[0] * scale_factors[0]),
            int(volume.shape[1] * scale_factors[1]),
            int(volume.shape[2] * scale_factors[2])
        ]
        
        # Resample using zoom
        resampled = ndimage.zoom(volume, scale_factors, order=3)
        
        print(f"✅ Volume resampled: {volume.shape} → {resampled.shape}")
        
        return resampled

print("📚 DICOMPreprocessor class defined!")

In [ ]:
def preprocess_dataset(ds_name, ds_path):
    """Preprocess a single dataset"""
    print(f"\n🚀 Preprocessing {ds_name}...")
    print(f"="*60)
    
    # Initialize preprocessor
    preprocessor = DICOMPreprocessor()
    
    try:
        # Load DICOM series
        volume, slice_positions = preprocessor.load_dicom_series(ds_path)
        
        # Read first DICOM for metadata
        dicom_files = [f for f in os.listdir(ds_path) if f.endswith('.dcm')]
        first_ds = pydicom.dcmread(os.path.join(ds_path, dicom_files[0]))
        
        # Convert to HU
        hu_volume = preprocessor.convert_to_hu(volume, first_ds)
        
        # Create stent mask
        stent_mask = preprocessor.create_stent_mask(
            hu_volume, 
            PIPELINE_CONFIG['preprocessing']['stent_hu_threshold']
        )
        
        # Normalize volume for processing
        normalized_volume = preprocessor.normalize_volume(hu_volume)
        
        # Save preprocessed data
        output_files = {
            'original': f'/content/preprocessed_data/{ds_name}_original.npy',
            'hu_volume': f'/content/preprocessed_data/{ds_name}_hu.npy',
            'normalized': f'/content/preprocessed_data/{ds_name}_normalized.npy',
            'mask': f'/content/preprocessed_data/{ds_name}_mask.npy',
            'spacing': f'/content/preprocessed_data/{ds_name}_spacing.npy',
            'metadata': f'/content/preprocessed_data/{ds_name}_metadata.json'
        }
        
        # Save numpy arrays
        np.save(output_files['original'], volume)
        np.save(output_files['hu_volume'], hu_volume)
        np.save(output_files['normalized'], normalized_volume)
        np.save(output_files['mask'], stent_mask)
        np.save(output_files['spacing'], np.array(preprocessor.spacing))
        
        # Save metadata
        metadata = {
            'dataset_name': ds_name,
            'original_shape': volume.shape,
            'spacing': preprocessor.spacing,
            'slice_positions': slice_positions,
            'hu_range': [float(hu_volume.min()), float(hu_volume.max())],
            'stent_voxels': int(np.sum(stent_mask)),
            'preprocessing_config': PIPELINE_CONFIG['preprocessing']
        }
        
        with open(output_files['metadata'], 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✅ {ds_name} preprocessing completed!")
        print(f"📁 Files saved to /content/preprocessed_data/")
        
        return {
            'volume': volume,
            'hu_volume': hu_volume,
            'normalized': normalized_volume,
            'mask': stent_mask,
            'spacing': preprocessor.spacing,
            'metadata': metadata
        }
        
    except Exception as e:
        print(f"❌ Error preprocessing {ds_name}: {e}")
        return None

print("📝 Preprocess function defined!")

In [ ]:
def preprocess_all_datasets():
    """Preprocess all three datasets"""
    print("🚀 Starting preprocessing of all datasets...")
    print("="*80)
    
    # Load dataset configuration
    with open('/content/pipeline_config.json', 'r') as f:
        config = json.load(f)
    
    # Dataset paths from previous setup
    dataset_paths = {
        'ds1': '/content/drive/MyDrive/aorta_p1_ds1',
        'ds2': '/content/drive/MyDrive/aorta_p1_ds2',
        'ds3': '/content/drive/MyDrive/aorta_p1_ds3'
    }
    
    results = {}
    
    for ds_name, ds_path in dataset_paths.items():
        print(f"\n📂 Processing {ds_name}...")
        
        if os.path.exists(ds_path):
            result = preprocess_dataset(ds_name, ds_path)
            if result:
                results[ds_name] = result
            else:
                print(f"⚠️ Failed to process {ds_name}")
        else:
            print(f"❌ Dataset path not found: {ds_path}")
            # Create synthetic data for demonstration
            print(f"🔧 Creating synthetic data for {ds_name}...")
            
            # Create synthetic volume
            shape = (128, 128, 50)
            synthetic_volume = np.random.normal(0, 50, shape)
            
            # Add synthetic aorta and stent
            center = (64, 64)
            for z in range(shape[2]):
                for i in range(-20, 21):
                    for j in range(-20, 21):
                        if i*i + j*j <= 400:  # Circle radius 20
                            x, y = center[0] + i, center[1] + j
                            if 0 <= x < 128 and 0 <= y < 128:
                                synthetic_volume[x, y, z] = np.random.normal(40, 10)  # Soft tissue
                
                # Add stent
                for i in range(-8, 9):
                    for j in range(-2, 3):
                        x, y = center[0] + i, center[1] + j
                        if 0 <= x < 128 and 0 <= y < 128:
                            synthetic_volume[x, y, z] = np.random.normal(2500, 100)  # Metal
            
            # Create synthetic mask
            synthetic_mask = np.zeros(shape, dtype=np.uint8)
            for z in range(shape[2]):
                for i in range(-8, 9):
                    for j in range(-2, 3):
                        x, y = center[0] + i, center[1] + j
                        if 0 <= x < 128 and 0 <= y < 128:
                            synthetic_mask[x, y, z] = 1
            
            # Save synthetic data
            np.save(f'/content/preprocessed_data/{ds_name}_original.npy', synthetic_volume)
            np.save(f'/content/preprocessed_data/{ds_name}_hu.npy', synthetic_volume)
            np.save(f'/content/preprocessed_data/{ds_name}_normalized.npy', 
                    (synthetic_volume - synthetic_volume.min()) / (synthetic_volume.max() - synthetic_volume.min()))
            np.save(f'/content/preprocessed_data/{ds_name}_mask.npy', synthetic_mask)
            np.save(f'/content/preprocessed_data/{ds_name}_spacing.npy', np.array([1.0, 1.0, 2.0]))
            
            results[ds_name] = {
                'volume': synthetic_volume,
                'hu_volume': synthetic_volume,
                'normalized': (synthetic_volume - synthetic_volume.min()) / (synthetic_volume.max() - synthetic_volume.min()),
                'mask': synthetic_mask,
                'spacing': (1.0, 1.0, 2.0),
                'metadata': {'dataset_name': ds_name, 'synthetic': True}
            }
            
            print(f"✅ Synthetic data created for {ds_name}")
    
    print(f"\n✅ All datasets preprocessed!")
    print(f"📊 Processed {len(results)} datasets")
    
    return results

# Run preprocessing
preprocessing_results = preprocess_all_datasets()

In [ ]:
def visualize_preprocessing_results(results):
    """Visualize preprocessing results"""
    if not results:
        print("⚠️ No preprocessing results to visualize!")
        return
    
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    
    for idx, (ds_name, data) in enumerate(results.items()):
        # Get middle slice
        middle_slice = data['volume'].shape[2] // 2
        
        # Original volume
        axes[idx, 0].imshow(data['volume'][:, :, middle_slice], cmap='gray')
        axes[idx, 0].set_title(f'{ds_name.upper()} - Original')
        axes[idx, 0].axis('off')
        
        # HU volume
        axes[idx, 1].imshow(data['hu_volume'][:, :, middle_slice], cmap='gray')
        axes[idx, 1].set_title(f'{ds_name.upper()} - HU Volume')
        axes[idx, 1].axis('off')
        
        # Normalized volume
        axes[idx, 2].imshow(data['normalized'][:, :, middle_slice], cmap='gray')
        axes[idx, 2].set_title(f'{ds_name.upper()} - Normalized')
        axes[idx, 2].axis('off')
        
        # Stent mask
        axes[idx, 3].imshow(data['volume'][:, :, middle_slice], cmap='gray')
        axes[idx, 3].imshow(data['mask'][:, :, middle_slice], alpha=0.5, cmap='Reds')
        axes[idx, 3].set_title(f'{ds_name.upper()} - Stent Mask')
        axes[idx, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/visualizations/preprocessing_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Preprocessing visualization completed!")

# Visualize results
visualize_preprocessing_results(preprocessing_results)

In [ ]:
def generate_preprocessing_report(results):
    """Generate preprocessing report"""
    print("\n" + "="*80)
    print("📊 PREPROCESSING REPORT")
    print("="*80)
    
    if not results:
        print("❌ No preprocessing results available!")
        return
    
    for ds_name, data in results.items():
        print(f"\n📂 {ds_name.upper()}:")
        print(f"  Volume shape: {data['volume'].shape}")
        print(f"  Spacing: {data['spacing']}")
        print(f"  HU range: [{data['hu_volume'].min():.1f}, {data['hu_volume'].max():.1f}]")
        print(f"  Stent voxels: {np.sum(data['mask'])}")
        print(f"  Stent volume: {np.sum(data['mask']) * np.prod(data['spacing']):.2f} mm³")
        
        if 'synthetic' in data['metadata'] and data['metadata']['synthetic']:
            print(f"  🔧 Synthetic data")
    
    print("\n✅ Preprocessing completed successfully!")
    print("📁 Data saved to /content/preprocessed_data/")
    print("📊 Visualizations saved to /content/visualizations/")
    print("\n🚀 Ready for EADTV enhancement!")
    print("="*80)

# Generate report
generate_preprocessing_report(preprocessing_results)